# Experimentos de desbalanceo en CGM basados en datos reales

Este notebook reconstruye escenarios experimentales a partir de los datasets reales utilizados en el EDA. El objetivo no es generar datos sinteticos desde cero, sino introducir modificaciones controladas sobre series reales para simular casuisticas clinicas plausibles: desbalanceos de distinta severidad, concentracion interpaciente de eventos, fragmentacion temporal, perdida de continuidad, cambios de frecuencia de muestreo, variaciones en la duracion de episodios y reduccion de cobertura longitudinal. Cada escenario representa una hipotesis metodologica derivada del informe de EDA y se documenta con justificacion clinica y analitica.

## Marco clinico y trazabilidad con el EDA

Las decisiones de diseno se basan en el informe de EDA. DIATREND muestra hipoglucemia extremadamente rara, episodios breves y alta concentracion de eventos en pocos pacientes; REPLACE-BG presenta episodios persistentes y autocorrelaciones altas; T1DiabetesGranada combina autocorrelacion alta con una fragmentacion temporal severa. Estos rasgos se traducen en escenarios experimentales con modificaciones puntuales que conservan la fisiologia medida por el CGM y reproducen los sesgos observados en el EDA.

## Setup y configuracion

Se establece un pipeline comun para cargar parquet reales, estandarizar columnas, seleccionar timestamps validos (5 min vs 15 min), etiquetar clases clinicas y construir episodios y segmentos temporales. Toda transformacion posterior se aplica sobre estos objetos estandarizados.

In [43]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(8):
        if (current / 'data').exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise FileNotFoundError('No se encontro la carpeta data/ desde el directorio actual.')

ROOT = find_project_root(Path.cwd())
DATASETS = {
    'DIATREND': "../../data/DIATREND/Glucose_measurements_DiaTrend_FILTERED_2026-03-27.parquet",
    'REPLACE-BG': "../../data/REPLACE-BG/Glucose_measurements_REPLACE-BG_FILTERED_2026-03-27.parquet",
    'T1DiabetesGranada': "../../data/T1DiabetesGranada/Glucose_measurements_T1DiabetesGranada_FILTERED_2026-03-27.parquet"
}

OUTPUT_DIR = ROOT / 'data' / 'processed' / 'experiments'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT detectado: {ROOT}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

ROOT detectado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments
OUTPUT_DIR: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments


## Revision explicita de schemas

Se valida el schema real de los parquet para evitar suposiciones. Se documentan tipos, nulos y rango temporal, y se deja trazabilidad de las columnas de timestamp disponibles para cada dataset.

In [44]:
def summarize_schema(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        dtype = str(s.dtype)
        nulls = int(s.isna().sum())
        null_pct = float(s.isna().mean() * 100)
        min_val, max_val = None, None
        if pd.api.types.is_numeric_dtype(s):
            min_val = s.min(skipna=True)
            max_val = s.max(skipna=True)
        elif pd.api.types.is_datetime64_any_dtype(s):
            min_val = s.min(skipna=True)
            max_val = s.max(skipna=True)
        rows.append({
            'dataset': name,
            'column': col,
            'dtype': dtype,
            'nulls': nulls,
            'null_pct': round(null_pct, 2),
            'min': min_val,
            'max': max_val,
            'n_unique': int(s.nunique(dropna=True))
        })
    return pd.DataFrame(rows)

schema_frames = []
for name, path in DATASETS.items():
    df = pd.read_parquet(path)
    schema_frames.append(summarize_schema(df, name))
schema_report = pd.concat(schema_frames, ignore_index=True)
display(schema_report)

display(schema_report.groupby('dataset', as_index=False)['null_pct'].mean().rename(columns={'null_pct': 'avg_null_pct'}))

,dataset,column,dtype,nulls,null_pct,min,max,n_unique
0,DIATREND,patient_id,object,0,0.00,None,None,51
1,DIATREND,measurement_date,object,1487733,16.25,None,None,2358
2,DIATREND,measurement_time,object,1487733,16.25,None,None,86400
3,DIATREND,measurement,Int16,1487733,16.25,39,401,363
4,DIATREND,15min,datetime64[ns],6238377,68.14,2015-12-01 21:00:00,2022-06-29 00:00:00,230509
5,DIATREND,5min,datetime64[ns],404964,4.42,2015-12-01 21:00:00,2022-06-29 00:00:00,691525
6,REPLACE-BG,patient_id,object,0,0.00,None,None,223
7,REPLACE-BG,measurement_date,object,1147000,9.91,None,None,318
8,REPLACE-BG,measurement_time,object,1147000,9.91,None,None,86400
9,REPLACE-BG,measurement,Int16,1147000,9.91,39,401,363


,dataset,avg_null_pct
0,DIATREND,20.218333
1,REPLACE-BG,16.163333
2,T1DiabetesGranada,30.390000


## Estandarizacion de series: timestamp, clases, episodios y segmentos

Se define un pipeline comun que selecciona el timestamp mas fiable, etiqueta la glucosa con umbrales clinicos y separa las series en segmentos continuos para evitar que gaps artificiales contaminen la deteccion de episodios. Estas decisiones reproducen los criterios del EDA y garantizan coherencia temporal.

In [45]:
def choose_timestamp_column(df: pd.DataFrame) -> tuple[str, int]:
    candidates = []
    for col, freq in [('5min', 5), ('15min', 15)]:
        if col in df.columns:
            null_pct = float(df[col].isna().mean())
            candidates.append((null_pct, col, freq))
    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][1], candidates[0][2]
    if {'measurement_date', 'measurement_time'}.issubset(df.columns):
        return 'measurement_date_time', 5
    raise ValueError('No se encontraron columnas temporales compatibles.')

def label_glucose(values: pd.Series) -> pd.Series:
    values = pd.to_numeric(values, errors='coerce')
    return pd.Series(np.select(
        [values < 70, values > 180],
        ['hypoglycemia', 'hyperglycemia'],
        default='normoglycemia',
    ), index=values.index)

def add_segments(df: pd.DataFrame, freq_min: int, gap_factor: float = 2.5) -> pd.DataFrame:
    df = df.sort_values(['patient_id', 'timestamp']).reset_index(drop=True)
    gap_limit = freq_min * gap_factor
    dt = df.groupby('patient_id', sort=False)['timestamp'].diff().dt.total_seconds().div(60)
    new_segment = dt.isna() | (dt > gap_limit)
    df['segment_id'] = new_segment.groupby(df['patient_id'], sort=False).cumsum()
    df['segment_id'] = df['patient_id'].astype(str) + '_s' + df['segment_id'].astype(int).astype(str)
    return df

def add_episode_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['patient_id', 'timestamp']).reset_index(drop=True)
    prev_label = df.groupby('patient_id', sort=False)['class_label'].shift(1)
    prev_segment = df.groupby('patient_id', sort=False)['segment_id'].shift(1)
    is_new_episode = (df['class_label'] != prev_label) | (df['segment_id'] != prev_segment)
    df['episode_num'] = is_new_episode.groupby(df['patient_id'], sort=False).cumsum()
    df['episode_id'] = df['patient_id'].astype(str) + '_e' + df['episode_num'].astype(int).astype(str) + '_' + df['class_label']
    df.drop(columns=['episode_num'], inplace=True)
    return df

def standardize_dataset(name: str, path: Path) -> tuple[pd.DataFrame, int]:
    df = pd.read_parquet(path)
    rename_map = {
        'patient_id': 'patient_id',
        'Patient_ID': 'patient_id',
        'measurement': 'glucose',
        'Measurement': 'glucose',
        'measurement_date': 'measurement_date',
        'Measurement_date': 'measurement_date',
        'measurement_time': 'measurement_time',
        'Measurement_time': 'measurement_time'
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    ts_col, freq_min = choose_timestamp_column(df)
    if ts_col == 'measurement_date_time':
        df['timestamp'] = pd.to_datetime(
            df['measurement_date'].astype(str) + ' ' + df['measurement_time'].astype(str),
            errors='coerce'
        )
    else:
        df['timestamp'] = pd.to_datetime(df[ts_col], errors='coerce')
    df['glucose'] = pd.to_numeric(df['glucose'], errors='coerce')
    df = df.dropna(subset=['patient_id', 'timestamp', 'glucose']).copy()
    df['patient_id'] = df['patient_id'].astype(str)
    df['class_label'] = label_glucose(df['glucose'])
    df['dataset'] = name
    df['freq_min'] = freq_min
    df = add_segments(df, freq_min=freq_min)
    df = add_episode_id(df)
    return df, freq_min

def recompute_structure(df: pd.DataFrame, freq_min: int) -> pd.DataFrame:
    df = df.sort_values(['patient_id', 'timestamp']).reset_index(drop=True)
    df = add_segments(df, freq_min=freq_min)
    df = add_episode_id(df)
    return df

def summarize_experiment(df: pd.DataFrame, label: str) -> pd.DataFrame:
    counts = df['class_label'].value_counts(normalize=True) * 100
    out = pd.DataFrame({'pct': counts}).reset_index().rename(columns={'index': 'class_label'})
    out['experiment'] = label
    return out

def save_experiment(df: pd.DataFrame, name: str) -> Path:
    out_path = OUTPUT_DIR / f'{name}.parquet'
    df.to_parquet(out_path, index=False)
    print(f'Guardado: {out_path}')
    return out_path

## Cohortes base para experimentacion

Se construyen cohortes base por dataset para mantener trazabilidad con el EDA y controlar el tamano de las series. La seleccion prioriza pacientes con suficiente continuidad temporal y registros suficientes para permitir modificaciones controladas sin colapsar la estructura de episodios.

In [46]:
def build_base_cohort(df: pd.DataFrame, max_patients: int, min_records: int, seed: int) -> pd.DataFrame:
    counts = df.groupby('patient_id', sort=False).size().reset_index(name='n')
    eligible = counts[counts['n'] >= min_records]['patient_id']
    if len(eligible) == 0:
        return df
    chosen = eligible.sample(min(max_patients, len(eligible)), random_state=seed).tolist()
    return df[df['patient_id'].isin(chosen)].copy()

standardized = {}
freq_map = {}
for name, path in DATASETS.items():
    df_std, freq_min = standardize_dataset(name, path)
    standardized[name] = df_std
    freq_map[name] = freq_min
    print(name, df_std.shape, 'freq_min', freq_min)

base_cohorts = {
    'DIATREND': build_base_cohort(standardized['DIATREND'], max_patients=40, min_records=4000, seed=SEED),
    'REPLACE-BG': build_base_cohort(standardized['REPLACE-BG'], max_patients=60, min_records=4000, seed=SEED),
    'T1DiabetesGranada': build_base_cohort(standardized['T1DiabetesGranada'], max_patients=80, min_records=3000, seed=SEED)
}

for name, df in base_cohorts.items():
    print(name, df.shape)
    display(summarize_experiment(df, f'base_{name}'))

DIATREND (7262392, 12) freq_min 5
REPLACE-BG (10381724, 12) freq_min 5
T1DiabetesGranada (21869345, 12) freq_min 15
DIATREND (5875335, 12)


,class_label,pct,experiment
0,normoglycemia,51.660952,base_DIATREND
1,hyperglycemia,46.282569,base_DIATREND
2,hypoglycemia,2.056478,base_DIATREND


REPLACE-BG (2763562, 12)


,class_label,pct,experiment
0,normoglycemia,62.962944,base_REPLACE-BG
1,hyperglycemia,32.963255,base_REPLACE-BG
2,hypoglycemia,4.073800,base_REPLACE-BG


T1DiabetesGranada (2434454, 12)


,class_label,pct,experiment
0,normoglycemia,60.806530,base_T1DiabetesGranada
1,hyperglycemia,34.900721,base_T1DiabetesGranada
2,hypoglycemia,4.292749,base_T1DiabetesGranada


In [47]:
def drop_episodes_to_target(df: pd.DataFrame, class_label: str, target_ratio: float, seed: int) -> pd.DataFrame:
    df = df.copy()
    n_total = len(df)
    n_class = int((df['class_label'] == class_label).sum())
    n_other = n_total - n_class
    if n_other == 0:
        return df
    n_target = int((target_ratio * n_other) / (1 - target_ratio))
    if n_target >= n_class:
        return df
    episodes = (
        df[df['class_label'] == class_label]
        .groupby('episode_id', sort=False)
        .size()
        .reset_index(name='len')
    )
    episodes = episodes.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    to_drop = []
    current = n_class
    for _, row in episodes.iterrows():
        if current <= n_target:
            break
        to_drop.append(row['episode_id'])
        current -= int(row['len'])
    return df[~df['episode_id'].isin(to_drop)].copy()

def drop_majority_to_target(df: pd.DataFrame, target_ratio: float, majority_label: str, seed: int) -> pd.DataFrame:
    df = df.copy()
    n_total = len(df)
    n_min = int((df['class_label'] == 'hypoglycemia').sum())
    if n_min == 0:
        return df
    n_maj = n_total - n_min
    n_maj_target = int((n_min * (1 - target_ratio)) / target_ratio)
    if n_maj_target >= n_maj:
        return df
    episodes = (
        df[df['class_label'] == majority_label]
        .groupby('episode_id', sort=False)
        .size()
        .reset_index(name='len')
    )
    episodes = episodes.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    to_drop = []
    current = n_maj
    for _, row in episodes.iterrows():
        if current <= n_maj_target:
            break
        to_drop.append(row['episode_id'])
        current -= int(row['len'])
    return df[~df['episode_id'].isin(to_drop)].copy()

def concentrate_events(df: pd.DataFrame, class_label: str, top_k: int) -> pd.DataFrame:
    df = df.copy()
    counts = df[df['class_label'] == class_label].groupby('patient_id').size().sort_values(ascending=False)
    keep_patients = set(counts.head(top_k).index)
    mask = (df['class_label'] != class_label) | (df['patient_id'].isin(keep_patients))
    return df[mask].copy()

def drop_time_blocks(df: pd.DataFrame, drop_frac: float, block_minutes: int, seed: int) -> pd.DataFrame:
    df = df.copy()
    rng_local = np.random.default_rng(seed)
    keep_rows = []
    for pid, g in df.groupby('patient_id', sort=False):
        g = g.sort_values('timestamp')
        if len(g) == 0:
            continue
        total_drop = int(len(g) * drop_frac)
        if total_drop <= 0:
            keep_rows.append(g)
            continue
        drop_mask = pd.Series(False, index=g.index)
        attempts = 0
        while drop_mask.sum() < total_drop and attempts < 50:
            start_idx = rng_local.integers(0, len(g))
            start_time = g.iloc[start_idx]['timestamp']
            end_time = start_time + pd.Timedelta(minutes=block_minutes)
            drop_mask |= (g['timestamp'] >= start_time) & (g['timestamp'] <= end_time)
            attempts += 1
        keep_rows.append(g[~drop_mask])
    return pd.concat(keep_rows, ignore_index=True)

def reduce_frequency(df: pd.DataFrame, step: int) -> pd.DataFrame:
    out = []
    for _, g in df.groupby(['patient_id', 'segment_id'], sort=False):
        g = g.sort_values('timestamp')
        out.append(g.iloc[::step])
    return pd.concat(out, ignore_index=True)

def truncate_episodes(df: pd.DataFrame, class_label: str, max_len: int) -> pd.DataFrame:
    df = df.copy()
    keep_idx = []
    for _, g in df.groupby('episode_id', sort=False):
        if g['class_label'].iloc[0] != class_label:
            keep_idx.append(g.index)
            continue
        keep_idx.append(g.sort_values('timestamp').head(max_len).index)
    keep_idx = np.concatenate(keep_idx)
    return df.loc[keep_idx].copy()

def keep_long_episodes(df: pd.DataFrame, class_label: str, min_len: int) -> pd.DataFrame:
    df = df.copy()
    lengths = df.groupby('episode_id', sort=False).size()
    long_eps = set(lengths[lengths >= min_len].index)
    mask = (df['class_label'] != class_label) | (df['episode_id'].isin(long_eps))
    return df[mask].copy()

def reduce_coverage(df: pd.DataFrame, days: int) -> pd.DataFrame:
    out = []
    for pid, g in df.groupby('patient_id', sort=False):
        g = g.sort_values('timestamp')
        start = g['timestamp'].min()
        end = start + pd.Timedelta(days=days)
        out.append(g[g['timestamp'] <= end])
    return pd.concat(out, ignore_index=True)

def reduce_patients(df: pd.DataFrame, n_patients: int, seed: int) -> pd.DataFrame:
    patients = df['patient_id'].drop_duplicates()
    if len(patients) <= n_patients:
        return df.copy()
    chosen = patients.sample(n_patients, random_state=seed)
    return df[df['patient_id'].isin(chosen)].copy()

## Experimento A — Imbalance leve (REPLACE-BG, aumento relativo de hipo por reduccion de normo)

Fenomeno: se simula un escenario de desbalanceo leve donde la hipoglucemia representa una fraccion moderada de la serie, sin introducir datos nuevos. Esto reproduce cohortes con monitorizacion intensiva donde los episodios hipoglucemicos se capturan con mas frecuencia, aumentando su presencia relativa.

Relevancia clinica: un incremento moderado de eventos puede alterar el umbral de decision de modelos predictivos y reducir falsos negativos a costa de mayor sensibilidad.

Motivacion EDA: REPLACE-BG exhibe episodios persistentes y autocorrelados, por lo que reducir segmentos largos de normoglucemia es plausible sin destruir la dinamica de crisis.

Hipotesis: tecnicas conservadoras como cost-sensitive learning y RUS temporal deberian comportarse de forma estable cuando el desbalanceo es leve.

In [48]:
expA = drop_majority_to_target(base_cohorts['REPLACE-BG'], target_ratio=0.06, majority_label='normoglycemia', seed=SEED)
expA = recompute_structure(expA, freq_map['REPLACE-BG'])
save_experiment(expA, 'expA_imbalance_leve_replacebg')
display(summarize_experiment(expA, 'expA'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expA_imbalance_leve_replacebg.parquet


,class_label,pct,experiment
0,hyperglycemia,48.550043,expA
1,normoglycemia,45.449846,expA
2,hypoglycemia,6.000111,expA


## Experimento B — Imbalance moderado (REPLACE-BG, referencia clinica)

Fenomeno: se mantiene un desbalanceo moderado similar al observado en la cohorte real, pero se fuerza el ratio de hipoglucemia para homogeneizar la severidad entre pacientes.

Relevancia clinica: en cohortes mas homogeneas los modelos suelen generalizar mejor, pero pueden ocultar heterogeneidad interpaciente real.

Motivacion EDA: REPLACE-BG presenta un ratio global de hipo cercano al 3.7% y alta autocorrelacion; el escenario prueba como responden las tecnicas cuando los episodios son frecuentes pero no extremos.

Hipotesis: tecnicas de oversampling episodico y SMOTE temporal deberian preservar mejor la dinamica en este entorno.

In [49]:
expB = drop_majority_to_target(base_cohorts['REPLACE-BG'], target_ratio=0.035, majority_label='normoglycemia', seed=SEED)
expB = recompute_structure(expB, freq_map['REPLACE-BG'])
save_experiment(expB, 'expB_imbalance_moderado_replacebg')
display(summarize_experiment(expB, 'expB'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expB_imbalance_moderado_replacebg.parquet


,class_label,pct,experiment
0,normoglycemia,62.962944,expB
1,hyperglycemia,32.963255,expB
2,hypoglycemia,4.073800,expB


## Experimento C — Imbalance severo (DIATREND, reduccion de episodios hipo)

Fenomeno: se reduce la presencia de hipoglucemia hasta ratios severos, replicando pacientes con eventos muy raros.

Relevancia clinica: los modelos tienden a infra-detectar hipoglucemias cuando la senal es extremadamente escasa.

Motivacion EDA: DIATREND presenta hipoglucemias breves y poco autocorreladas; el escenario intensifica este patron para analizar sensibilidad de tecnicas de resampling.

Hipotesis: las tecnicas basadas en episodios completos evitaran artefactos respecto a tecnicas puntuales.

In [50]:
expC = drop_episodes_to_target(base_cohorts['DIATREND'], class_label='hypoglycemia', target_ratio=0.015, seed=SEED)
expC = recompute_structure(expC, freq_map['DIATREND'])
save_experiment(expC, 'expC_imbalance_severo_diatrend')
display(summarize_experiment(expC, 'expC'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expC_imbalance_severo_diatrend.parquet


,class_label,pct,experiment
0,normoglycemia,51.954534,expC
1,hyperglycemia,46.545587,expC
2,hypoglycemia,1.499880,expC


## Experimento D — Imbalance extremo (DIATREND, hipoglucemia casi ausente)

Fenomeno: se simula un ratio extremo de hipoglucemia cercano a los casos mas raros del EDA.

Relevancia clinica: representa pacientes donde las crisis hipoglucemicas son excepcionales, dificultando la calibracion de alarmas.

Motivacion EDA: en DIATREND existen pacientes con ratios superiores a 1:1000; este escenario reproduce ese sesgo.

Hipotesis: el balanceo por paciente sera mas estable que cualquier tecnica global.

In [51]:
expD = drop_episodes_to_target(base_cohorts['DIATREND'], class_label='hypoglycemia', target_ratio=0.002, seed=SEED)
expD = recompute_structure(expD, freq_map['DIATREND'])
save_experiment(expD, 'expD_imbalance_extremo_diatrend')
display(summarize_experiment(expD, 'expD'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expD_imbalance_extremo_diatrend.parquet


,class_label,pct,experiment
0,normoglycemia,52.640302,expD
1,hyperglycemia,47.159960,expD
2,hypoglycemia,0.199739,expD


## Experimento E — Concentracion de hipoglucemia en pocos pacientes (DIATREND)

Fenomeno: se conserva la hipoglucemia solo en un subconjunto pequeno de pacientes con alta carga de eventos.

Relevancia clinica: reproduce sesgos de entrenamiento donde el modelo aprende patrones de pocos individuos y pierde generalizacion.

Motivacion EDA: el top 5 de pacientes concentra mas del 50% de eventos hipo en DIATREND.

Hipotesis: el balanceo adaptativo por paciente reduce este sesgo mejor que estrategias globales.

In [52]:
expE = concentrate_events(base_cohorts['DIATREND'], class_label='hypoglycemia', top_k=5)
expE = recompute_structure(expE, freq_map['DIATREND'])
save_experiment(expE, 'expE_concentracion_hypo_diatrend')
display(summarize_experiment(expE, 'expE'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expE_concentracion_hypo_diatrend.parquet


,class_label,pct,experiment
0,normoglycemia,52.102892,expE
1,hyperglycemia,46.678500,expE
2,hypoglycemia,1.218608,expE


## Experimento F — Heterogeneidad interpaciente controlada (DIATREND)

Fenomeno: se fuerza un escenario donde coexisten pacientes con ratios extremos, severos y moderados mediante reduccion diferenciada de episodios.

Relevancia clinica: refleja la variabilidad real en riesgo hipoglucemico, clave para evaluar estrategias por paciente.

Motivacion EDA: el ratio de desbalanceo vario entre 1:10 y 1:1376 en DIATREND.

Hipotesis: los modelos con pesos globales tenderan a sesgarse hacia subgrupos dominantes.

In [53]:
def apply_patient_targets(df: pd.DataFrame, targets: dict[str, float], seed: int) -> pd.DataFrame:
    out = []
    for pid, g in df.groupby('patient_id', sort=False):
        target = targets.get(pid, None)
        if target is None:
            out.append(g)
            continue
        g2 = drop_episodes_to_target(g, class_label='hypoglycemia', target_ratio=target, seed=seed)
        out.append(g2)
    return pd.concat(out, ignore_index=True)

patients = base_cohorts['DIATREND']['patient_id'].drop_duplicates().tolist()
rng_local = np.random.default_rng(SEED)
rng_local.shuffle(patients)
n = len(patients)
targets = {}
for i, pid in enumerate(patients):
    if i < n * 0.2:
        targets[pid] = 0.002
    elif i < n * 0.6:
        targets[pid] = 0.01
    else:
        targets[pid] = 0.04
expF = apply_patient_targets(base_cohorts['DIATREND'], targets, seed=SEED)
expF = recompute_structure(expF, freq_map['DIATREND'])
save_experiment(expF, 'expF_heterogeneidad_paciente_diatrend')
display(summarize_experiment(expF, 'expF'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expF_heterogeneidad_paciente_diatrend.parquet


,class_label,pct,experiment
0,normoglycemia,52.088997,expF
1,hyperglycemia,46.666051,expF
2,hypoglycemia,1.244952,expF


## Experimento G — Fragmentacion temporal (T1DiabetesGranada)

Fenomeno: se eliminan bloques temporales continuos para inducir gaps artificiales y aumentar la fragmentacion.

Relevancia clinica: la perdida de continuidad impide aprender transiciones completas y deteriora la autocorrelacion.

Motivacion EDA: T1DiabetesGranada exhibe fragmentacion extrema con muy pocas ventanas largas.

Hipotesis: las tecnicas que respetan segmentos continuos evitaran construir episodios falsos.

In [54]:
expG = drop_time_blocks(base_cohorts['T1DiabetesGranada'], drop_frac=0.25, block_minutes=180, seed=SEED)
expG = recompute_structure(expG, freq_map['T1DiabetesGranada'])
save_experiment(expG, 'expG_fragmentacion_t1dgranada')
display(summarize_experiment(expG, 'expG'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expG_fragmentacion_t1dgranada.parquet


,class_label,pct,experiment
0,normoglycemia,60.792459,expG
1,hyperglycemia,34.909156,expG
2,hypoglycemia,4.298385,expG


## Experimento H — Perdida de continuidad intra-episodio (T1DiabetesGranada)

Fenomeno: se eliminan mediciones dentro de segmentos continuos para simular ausencias puntuales del CGM sin generar nuevos datos.

Relevancia clinica: los huecos cortos degradan la estimacion de tasas de cambio y la deteccion de pendientes rapidas.

Motivacion EDA: el dataset contiene gaps frecuentes; este escenario focaliza en faltantes locales mas sutiles.

Hipotesis: las tecnicas basadas en episodios completos seran mas robustas a estos huecos que las basadas en puntos.

In [55]:
expH = base_cohorts['T1DiabetesGranada'].copy()
expH = expH.sample(frac=0.85, random_state=SEED)
expH = recompute_structure(expH, freq_map['T1DiabetesGranada'])
save_experiment(expH, 'expH_perdida_continuidad_t1dgranada')
display(summarize_experiment(expH, 'expH'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expH_perdida_continuidad_t1dgranada.parquet


,class_label,pct,experiment
0,normoglycemia,60.808849,expH
1,hyperglycemia,34.901459,expH
2,hypoglycemia,4.289692,expH


## Experimento I — Reduccion de frecuencia de muestreo (T1DiabetesGranada)

Fenomeno: se disminuye la frecuencia efectiva de muestreo manteniendo solo cada segundo punto en segmentos continuos.

Relevancia clinica: una frecuencia menor puede ocultar descensos rapidos y subestimar la velocidad de cambio.

Motivacion EDA: T1DiabetesGranada opera mayoritariamente a 15 min; se simula una degradacion adicional.

Hipotesis: tecnicas de balanceo que dependen de ventanas cortas sufriran mayor perdida de informacion.

In [56]:
expI = reduce_frequency(base_cohorts['T1DiabetesGranada'], step=2)
expI = recompute_structure(expI, freq_map['T1DiabetesGranada'] * 2)
expI['freq_min'] = expI['freq_min'] * 2
save_experiment(expI, 'expI_reduccion_frecuencia_t1dgranada')
display(summarize_experiment(expI, 'expI'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expI_reduccion_frecuencia_t1dgranada.parquet


,class_label,pct,experiment
0,normoglycemia,60.790626,expI
1,hyperglycemia,34.914339,expI
2,hypoglycemia,4.295035,expI


## Experimento J — Episodios hipoglucemicos cortos (REPLACE-BG)

Fenomeno: se truncan episodios hipoglucemicos largos para simular crisis cortas y aisladas.

Relevancia clinica: episodios breves dificultan la anticipacion porque la ventana de senal es limitada.

Motivacion EDA: DIATREND presenta episodios medianos de 4 pasos, muy inferiores a REPLACE-BG.

Hipotesis: los metodos que dependen de contextos largos se degradaran en este escenario.

In [57]:
expJ = truncate_episodes(base_cohorts['REPLACE-BG'], class_label='hypoglycemia', max_len=4)
expJ = recompute_structure(expJ, freq_map['REPLACE-BG'])
save_experiment(expJ, 'expJ_episodios_cortos_replacebg')
display(summarize_experiment(expJ, 'expJ'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expJ_episodios_cortos_replacebg.parquet


,class_label,pct,experiment
0,normoglycemia,64.486696,expJ
1,hyperglycemia,33.760991,expJ
2,hypoglycemia,1.752313,expJ


## Experimento K — Episodios hipoglucemicos largos (REPLACE-BG)

Fenomeno: se conservan solo episodios hipoglucemicos con duracion prolongada, eliminando episodios cortos.

Relevancia clinica: episodios largos amplifican la autocorrelacion y pueden sesgar el entrenamiento hacia patrones persistentes.

Motivacion EDA: REPLACE-BG muestra episodios mas largos y autocorrelados que DIATREND.

Hipotesis: tecnicas basadas en episodios completos mostraran mayor estabilidad aqui que en episodios cortos.

In [58]:
expK = keep_long_episodes(base_cohorts['REPLACE-BG'], class_label='hypoglycemia', min_len=12)
expK = recompute_structure(expK, freq_map['REPLACE-BG'])
save_experiment(expK, 'expK_episodios_largos_replacebg')
display(summarize_experiment(expK, 'expK'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expK_episodios_largos_replacebg.parquet


,class_label,pct,experiment
0,normoglycemia,64.263961,expK
1,hyperglycemia,33.644382,expK
2,hypoglycemia,2.091657,expK


## Experimento L — Reduccion de cobertura longitudinal (T1DiabetesGranada)

Fenomeno: se limita la serie a una ventana temporal corta por paciente, simulando cohortes con cobertura breve.

Relevancia clinica: la falta de seguimiento prolongado reduce la exposicion a eventos raros y altera la distribucion de episodios.

Motivacion EDA: la cobertura efectiva en T1DiabetesGranada es baja cuando se exige continuidad.

Hipotesis: los modelos entrenados con poca cobertura seran mas sensibles al ruido y al sesgo de paciente.

In [59]:
expL = reduce_coverage(base_cohorts['T1DiabetesGranada'], days=7)
expL = recompute_structure(expL, freq_map['T1DiabetesGranada'])
save_experiment(expL, 'expL_cobertura_reducida_t1dgranada')
display(summarize_experiment(expL, 'expL'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expL_cobertura_reducida_t1dgranada.parquet


,class_label,pct,experiment
0,normoglycemia,61.846970,expL
1,hyperglycemia,32.836361,expL
2,hypoglycemia,5.316669,expL


## Experimento M — Reduccion del tamano muestral (REPLACE-BG)

Fenomeno: se reduce el numero de pacientes manteniendo series completas de los seleccionados.

Relevancia clinica: cohortes pequenas amplifican sesgos individuales y dificultan la generalizacion.

Motivacion EDA: REPLACE-BG es el dataset con mas pacientes; este escenario prueba sensibilidad a tamanos menores.

Hipotesis: los metodos de balanceo que dependen de diversidad poblacional perderan estabilidad.

In [60]:
expM = reduce_patients(base_cohorts['REPLACE-BG'], n_patients=20, seed=SEED)
expM = recompute_structure(expM, freq_map['REPLACE-BG'])
save_experiment(expM, 'expM_reduccion_muestral_replacebg')
display(summarize_experiment(expM, 'expM'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expM_reduccion_muestral_replacebg.parquet


,class_label,pct,experiment
0,normoglycemia,59.207849,expM
1,hyperglycemia,36.961593,expM
2,hypoglycemia,3.830558,expM


## Experimento N — Escenario hibrido (T1DiabetesGranada)

Fenomeno: combina desbalanceo severo, fragmentacion temporal y concentracion parcial de eventos, simulando una cohorte clinica compleja con multiples sesgos simultaneos.

Relevancia clinica: escenarios reales suelen mezclar varios factores de sesgo, no solo un unico fenomeno aislado.

Motivacion EDA: T1DiabetesGranada incorpora fragmentacion y heterogeneidad, y DIATREND aporta la concentracion extrema de eventos.

Hipotesis: solo tecnicas que preserven episodios y respeten segmentos continuos mantendran coherencia fisiologica.

In [61]:
expN = drop_episodes_to_target(base_cohorts['T1DiabetesGranada'], class_label='hypoglycemia', target_ratio=0.01, seed=SEED)
expN = drop_time_blocks(expN, drop_frac=0.2, block_minutes=120, seed=SEED)
expN = concentrate_events(expN, class_label='hypoglycemia', top_k=10)
expN = recompute_structure(expN, freq_map['T1DiabetesGranada'])
save_experiment(expN, 'expN_hibrido_t1dgranada')
display(summarize_experiment(expN, 'expN'))

Guardado: C:\Users\User\Desktop\TFM-Glucose-Prediction\notebooks\04_Experiments\data\processed\experiments\expN_hibrido_t1dgranada.parquet


,class_label,pct,experiment
0,normoglycemia,63.221397,expN
1,hyperglycemia,36.272789,expN
2,hypoglycemia,0.505815,expN


## Catalogo de escenarios generados

Se listan los experimentos producidos, con sus rutas de salida. Estos archivos son la base para el notebook de tecnicas de balanceo.

In [62]:
catalog = pd.DataFrame([
    {'experiment': 'expA_imbalance_leve_replacebg', 'path': str(OUTPUT_DIR / 'expA_imbalance_leve_replacebg.parquet')},
    {'experiment': 'expB_imbalance_moderado_replacebg', 'path': str(OUTPUT_DIR / 'expB_imbalance_moderado_replacebg.parquet')},
    {'experiment': 'expC_imbalance_severo_diatrend', 'path': str(OUTPUT_DIR / 'expC_imbalance_severo_diatrend.parquet')},
    {'experiment': 'expD_imbalance_extremo_diatrend', 'path': str(OUTPUT_DIR / 'expD_imbalance_extremo_diatrend.parquet')},
    {'experiment': 'expE_concentracion_hypo_diatrend', 'path': str(OUTPUT_DIR / 'expE_concentracion_hypo_diatrend.parquet')},
    {'experiment': 'expF_heterogeneidad_paciente_diatrend', 'path': str(OUTPUT_DIR / 'expF_heterogeneidad_paciente_diatrend.parquet')},
    {'experiment': 'expG_fragmentacion_t1dgranada', 'path': str(OUTPUT_DIR / 'expG_fragmentacion_t1dgranada.parquet')},
    {'experiment': 'expH_perdida_continuidad_t1dgranada', 'path': str(OUTPUT_DIR / 'expH_perdida_continuidad_t1dgranada.parquet')},
    {'experiment': 'expI_reduccion_frecuencia_t1dgranada', 'path': str(OUTPUT_DIR / 'expI_reduccion_frecuencia_t1dgranada.parquet')},
    {'experiment': 'expJ_episodios_cortos_replacebg', 'path': str(OUTPUT_DIR / 'expJ_episodios_cortos_replacebg.parquet')},
    {'experiment': 'expK_episodios_largos_replacebg', 'path': str(OUTPUT_DIR / 'expK_episodios_largos_replacebg.parquet')},
    {'experiment': 'expL_cobertura_reducida_t1dgranada', 'path': str(OUTPUT_DIR / 'expL_cobertura_reducida_t1dgranada.parquet')},
    {'experiment': 'expM_reduccion_muestral_replacebg', 'path': str(OUTPUT_DIR / 'expM_reduccion_muestral_replacebg.parquet')},
    {'experiment': 'expN_hibrido_t1dgranada', 'path': str(OUTPUT_DIR / 'expN_hibrido_t1dgranada.parquet')}
])
display(catalog)

,experiment,path
0,expA_imbalance_leve_replacebg,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
1,expB_imbalance_moderado_replacebg,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
2,expC_imbalance_severo_diatrend,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
3,expD_imbalance_extremo_diatrend,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
4,expE_concentracion_hypo_diatrend,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
5,expF_heterogeneidad_paciente_diatrend,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
6,expG_fragmentacion_t1dgranada,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
7,expH_perdida_continuidad_t1dgranada,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
8,expI_reduccion_frecuencia_t1dgranada,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
9,expJ_episodios_cortos_replacebg,C:\Users\User\Desktop\TFM-Glucose-Prediction\n...
